In [ ]:
#Импорт необходимых библиотек и инструментов
import pandas as pd
import numpy as np
import re

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split


In [ ]:
from google.colab import files
uploaded = files.upload()

#для воспроизводимости кода нужно каждый раз при открытии кода нужно загружать файлы train.xlsx и test.xlsx

Saving test.xlsx to test (1).xlsx
Saving train.xlsx to train (1).xlsx


In [ ]:
#Загрузка файлов
train_df = pd.read_excel('train.xlsx')
test_df = pd.read_excel('test.xlsx')
train_df.head()

,Количество комнат,Тип,Метро,Адрес,"Площадь, м2",Дом,Парковка,Описание,Ремонт,"Площадь комнат, м2",...,Санузел,Название ЖК,Серия дома,"Высота потолков, м",Лифт,Мусоропровод,Доля,Валюта,Сведения об условиях,Стоимость
0,"3, Изолированная",Продажа квартиры,м. Пионерская (3 мин пешком),"Санкт-Петербург, Коломяжский проспект, 26",90.4/58.5/12.0,"10/24, Монолитный",подземная,Представьте квартиру вашей мечты. Она уже суще...,Евроремонт,NaN,...,Раздельный (2),NaN,NaN,2.75,"Пасс (1), Груз (1)",Да,NaN,RUB,"Свободная продажа, Возможна ипотека",24800000
1,"2, Оба варианта",Продажа квартиры,м. Фили (4 мин пешком),"Москва, Багратионовский проезд, 5Ак1",43.0/26.0/12.0,"5/6, Монолитный",подземная,Продается нежная квартира с уютом и встроенной...,Дизайнерский,NaN,...,Совмещенный (1),"Filicity, Корпус - Багратионовский, 5Ак1 (корп...",NaN,3.10,"Пасс (1), Груз (1)",NaN,NaN,RUB,"Свободная продажа, Возможна ипотека",29000000
2,"3, Оба варианта",Продажа квартиры,м. Геологическая (13 мин пешком),"Свердловская область, Екатеринбург, улица Сакк...",125.7/32.4/18.6,"6/11, Монолитно-кирпичный",подземная,Предлагаю просторную квартиру на 6- этаже в ле...,Дизайнерский,NaN,...,Совмещенный (3),NaN,NaN,3.00,Пасс (1),NaN,NaN,RUB,"Свободная продажа, Возможна ипотека",57500000
3,1,Продажа квартиры,м. Рязанский проспект (17 мин пешком),"Москва, улица Академика Скрябина, 26К5",31.7/21.5/5.2,"4/9, Панельный",NaN,Арт. 128732248 Уникальное предложение: уютная ...,Косметический,21.5,...,Совмещенный (1),NaN,NaN,NaN,Пасс (1),NaN,NaN,RUB,"Свободная продажа, Возможна ипотека",9800000
4,"3, Оба варианта",Продажа квартиры,м. Динамо (4 мин на машине),"Свердловская область, Екатеринбург, улица Сове...",58.4/42.6/6.7,"3/5, Панельный",наземная,"Тихая теплая квартира, удобная для жизни, тран...",Косметический,NaN,...,Совмещенный (1),NaN,NaN,2.50,NaN,NaN,NaN,RUB,"Свободная продажа, Возможна ипотека",6400000


In [ ]:
#Обработка данных и подготовка признаков для модели
def extract_rooms(x): #функция, позволяющая извлечь количество комнат из исходного текста (для апартаментов и студий функция будет возвращать 0)
    if pd.isna(x): return 0
    match = re.search(r'(\d+)', str(x))
    return int(match.group(1)) if match else 0

def extract_floors(x): #функция, позволяющая извлечь этаж и общее количество этажей в доме из исходного текста, в случае, если данные не будут найдены, вернется NaN
    if pd.isna(x): return (np.nan, np.nan)
    match = re.search(r'(\d+)/(\d+)', str(x))
    if match: return (int(match.group(1)), int(match.group(2)))
    return (np.nan, np.nan)

def extract_areas(x): #функция, позволяющая извлечь площадь всей квартиры, жилых зон и кухни по отдельности, в случае отсутствия данных вернется NaN
    if pd.isna(x): return (np.nan, np.nan, np.nan)
    parts = str(x).split('/')
    total = float(parts[0]) if len(parts) >= 1 else np.nan #общая площадь
    living = float(parts[1]) if len(parts) >= 2 else np.nan #жилая площадь
    kitchen = float(parts[2]) if len(parts) >= 3 else np.nan #площадь кухни
    return (total, living, kitchen)


def extract_renovation(x): #функция, позволяющая извлечь тип ремонта в квартире, и потом обращаться к нему по ключевому слову, в случае отсутствия данных вернется unknown
    if pd.isna(x): return 'unknown'
    text = str(x).lower()
    if 'без' in text: return 'no'
    if 'косметический' in text: return 'cosmetic'
    if 'евро' in text: return 'euro'
    if 'дизайнерский' in text: return 'design'
    return 'unknown'#в случае если ни один из перечисленных выше вариантов не подходит

def extract_city(x): #функция, позволяющая извлечь название города из выборки
    if pd.isna(x): return 'other'
    text = str(x).lower()
    if 'москва' in text: return 'moscow'
    if 'петербург' in text: return 'spb'
    if 'екатеринбург' in text or 'свердлов' in text: return 'ekb'
    return 'other' #в случае если ни один из перечисленных выше городов не подходит

def extract_house_type(x): #функция, позволяющая извлечь тип дома, в котором находится квартира
    if pd.isna(x): return 'unknown'
    text = str(x).lower()
    if 'кирпичный' in text: return 'brick'
    if 'монолитно-кирпичный' in text or 'кирпично-монолитный' in text: return 'monolith_brick'
    if 'монолитный' in text: return 'monolith'
    if 'панельный' in text: return 'panel'
    if 'блочный' in text: return 'block'
    return 'unknown' #в случае если ни один из перечисленных выше типов не подходит

def preprocess(df, is_train=True):
    df = df.copy()

    if is_train:
        y = df['Стоимость'].copy()
        df = df.drop('Стоимость', axis=1)
    else:
        y = None

    #Количество комнат
    df['rooms'] = df['Количество комнат'].apply(extract_rooms)

    #Этажи
    df[['floor', 'total_floors']] = df['Дом'].apply(extract_floors).apply(pd.Series)
    df['floor_ratio'] = df['floor'] / df['total_floors']

    #Площади
    df[['total_area', 'living_area', 'kitchen_area']] = df['Площадь, м2'].apply(extract_areas).apply(pd.Series)

    #Бинарные признаки, при наличии которого будет выдавать 1, при отсутствии - 0
    df['has_balcony'] = df['Балкон'].notna().astype(int)
    df['has_parking'] = df['Парковка'].notna().astype(int)
    df['has_elevator'] = df['Лифт'].notna().astype(int)
    df['has_trash'] = df['Мусоропровод'].notna().astype(int)
    df['is_share'] = df['Доля'].notna().astype(int)
    df['is_usd'] = (df['Валюта'] == 'USD').astype(int)

    #Категориальные признаки
    df['renovation'] = df['Ремонт'].apply(extract_renovation)
    df['city'] = df['Адрес'].apply(extract_city)
    df['house_type'] = df['Дом'].apply(extract_house_type)

    #Высота потолков
    df['ceiling_height'] = pd.to_numeric(df['Высота потолков, м'], errors='coerce')

    #Удаление исходных колонок
    cols_to_drop = ['Количество комнат', 'Тип', 'Метро', 'Адрес', 'Площадь, м2', 'Дом',
                    'Парковка', 'Описание', 'Ремонт', 'Площадь комнат, м2', 'Балкон',
                    'Окна', 'Санузел', 'Название ЖК', 'Серия дома', 'Высота потолков, м',
                    'Лифт', 'Мусоропровод', 'Доля', 'Валюта', 'Сведения об условиях']
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors='ignore')

    return df, y







In [ ]:
X_train, y_train = preprocess(train_df, is_train=True)
X_test, _ = preprocess(test_df, is_train=False)

X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=228
)

#Типы признаков
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

#Пайплайн с RobustScaler, использую его, так как он наиболее устойчив к выбросам
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')), # использую медианное значение, так как оно более устойчиво к выбросам, чем среднее
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(sparse_output=False)) #возвращает обычный numpy массив
])

preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

#Градиентный бустинг
model = Pipeline([
    ('pre', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=350,
        learning_rate=0.05,
        max_depth=4,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.8,
        random_state=228,
        loss='huber' # использую эту функцию, так как она сочетает в себе свойства MAE и MSE, следовательно устойчива к выбросам
    ))
])

#Обучение на тренировочных данных
model.fit(X_train_part, y_train_part)

#Предсказание на валидационной выборке
val_predictions = model.predict(X_val)

#Расчет MAE на валидации
val_mae = mean_absolute_error(y_val, val_predictions)
print(f"MAE на валидации = {val_mae:,.2f} руб.")

#Расчет MAE на обучении
train_predictions = model.predict(X_train_part)
train_mae = mean_absolute_error(y_train_part, train_predictions)
print(f"MAE на обучении = {train_mae:,.2f} руб.")

#Предсказания на тесте
test_predictions = model.predict(X_test)


MAE на валидации = 6,397,797.79 руб.
MAE на обучении = 4,948,405.24 руб.


In [ ]:
#Создание submission

submission = pd.DataFrame({
    'id': range(len(test_predictions)),
    'Стоимость': test_predictions.astype(int)  #округляем до целых чисел
})
submission.to_csv('submission.csv', index=False)
